# Chapter 2 — Math Building Blocks: Tensors, Gradient Descent, First Neural Net

Maps to Chollet Ch.2. Goal: understand **tensors**, **tensor ops**, and **how a net learns**
(gradient descent + backprop) — then read your first Keras model line by line.

> Backend note: Keras 3 needs a backend. On Colab it's preinstalled. Locally:
> `pip install keras tensorflow-cpu` (or jax/torch).

## 1. The "Hello World": classify MNIST digits in ~15 lines
Run it first, understand it last. Each piece is dissected in §5.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")  # or "jax" / "torch"
import keras
from keras import layers
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print("train:", train_images.shape, train_images.dtype, "| labels:", train_labels.shape)

# preprocess: flatten 28x28 -> 784, scale 0..255 -> 0..1, float32
train_x = train_images.reshape(60000, 28*28).astype("float32") / 255
test_x  = test_images.reshape(10000, 28*28).astype("float32") / 255

model = keras.Sequential([
    layers.Dense(512, activation="relu"),
    layers.Dense(10,  activation="softmax"),   # 10 probs summing to 1
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",  # integer labels -> use 'sparse'
              metrics=["accuracy"])

model.fit(train_x, train_labels, epochs=5, batch_size=128)
test_loss, test_acc = model.evaluate(test_x, test_labels, verbose=0)
print(f"test_acc: {test_acc:.4f}")   # ~0.978; train acc is higher -> that gap is OVERFITTING


In [ ]:
# Predicting: model outputs a probability vector per image; argmax = predicted class
probs = model.predict(test_x[:5], verbose=0)
print("prob vector for img0 (rounded):", probs[0].round(3))
print("predicted:", probs.argmax(axis=1), "| true:", test_labels[:5])


## 2. Tensors = N-dimensional arrays
A tensor has 3 defining attributes you check constantly:
- **rank / ndim** = number of axes
- **shape** = size along each axis (a tuple)
- **dtype** = element type (`float32` for model inputs, almost always)

| rank | name | example shape | example |
|---|---|---|---|
| 0 | scalar | `()` | `5` |
| 1 | vector | `(5,)` | one sample's features |
| 2 | matrix | `(samples, features)` | a tabular batch |
| 3 | — | `(samples, timesteps, features)` | timeseries / text |
| 4 | — | `(samples, height, width, channels)` | images |
| 5 | — | `(samples, frames, h, w, channels)` | video |

**Axis 0 is (almost) always the samples/batch axis.** A *batch* is a slice along axis 0.

In [ ]:
import numpy as np
s = np.array(12)                       # scalar
v = np.array([12, 3, 6, 14, 7])        # vector
m = np.array([[5,78,2],[6,79,3],[7,80,4]])   # matrix
for name,t in [("scalar",s),("vector",v),("matrix",m)]:
    print(f"{name:7s} ndim={t.ndim}  shape={t.shape}  dtype={t.dtype}")

# MNIST is rank-3: 60000 images of 28x28
print("\ntrain_images:", train_images.ndim, train_images.shape, train_images.dtype)

# slicing = selecting along axes. batch n of size 128:
n = 3
batch = train_images[128*n : 128*(n+1)]
print("batch:", batch.shape)
# crop center 14x14 of every image (negative indices ok):
print("center crop:", train_images[:, 7:-7, 7:-7].shape)


## 3. Tensor operations = the gears
A `Dense` layer computes exactly this: `output = relu(W @ x + b)`. Three ops:
**element-wise**, **broadcasting**, **dot product**. Plus **reshape**.

In [ ]:
import numpy as np
x = np.array([[1.,2.,3.],[4.,5.,6.]])     # (2,3)

# element-wise (applied per element)
print("relu:\n", np.maximum(x, 0))         # relu(x) = max(x,0)
print("x*2 + 1:\n", x*2 + 1)

# broadcasting: smaller tensor is virtually repeated to match shapes
b = np.array([10., 20., 30.])              # (3,) adds to each row of (2,3)
print("x + b:\n", x + b)

# dot product (matrix multiply) with @ ; shapes must align: (2,3)@(3,4)->(2,4)
W = np.ones((3,4))
print("x @ W shape:", (x @ W).shape)

# reshape: rearrange elements (count must match); -1 = infer
print("reshape:", np.arange(6).reshape(2,3).reshape(-1))

# one Dense layer, by hand:
def dense(x, W, b): return np.maximum(x @ W + b, 0)
print("manual dense out shape:", dense(x, np.ones((3,5)), np.zeros(5)).shape)


## 4. How a net learns: gradient descent + backprop
Training = find weights that minimize the **loss**. The recipe:

1. **derivative** of loss wrt a weight = slope = "which way is uphill, how steep".
2. **gradient** = the vector of all those derivatives (one per weight).
3. **step** weights *opposite* the gradient: `w ← w − lr · gradient`. `lr` = learning rate.
4. **backpropagation** = chain rule, computed automatically, to get every gradient in one backward pass.
5. **epoch** = one full pass over the data; **batch** = the chunk used per step (mini-batch SGD).

You're an algorithmist — so here is gradient descent with **no framework**, fitting `y = 2x + 1`.
Watch `w,b` march toward `2,1` and the loss fall.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.uniform(-1, 1, size=200)
y = 2*X + 1 + rng.normal(0, 0.1, size=200)     # true line + noise

w, b, lr = 0.0, 0.0, 0.1                         # start at zero
for step in range(101):
    pred = w*X + b
    loss = np.mean((pred - y)**2)                # MSE
    # gradients of MSE wrt w and b (the chain rule, done by hand here)
    grad_w = np.mean(2*(pred - y)*X)
    grad_b = np.mean(2*(pred - y))
    w -= lr*grad_w                               # the descent step
    b -= lr*grad_b
    if step % 20 == 0:
        print(f"step {step:3d}  loss={loss:.4f}  w={w:.3f}  b={b:.3f}")
print(f"\nlearned w≈{w:.3f} (true 2), b≈{b:.3f} (true 1)")


In [ ]:
# Same idea, but let the framework compute the gradient for you (autodiff).
# This is literally what model.fit() does under the hood, scaled up.
import os; os.environ.setdefault("KERAS_BACKEND","tensorflow")
import tensorflow as tf
Xt = tf.constant(X, tf.float32); yt = tf.constant(y, tf.float32)
w = tf.Variable(0.0); b = tf.Variable(0.0); lr = 0.1
for step in range(101):
    with tf.GradientTape() as tape:                 # records ops for autodiff
        loss = tf.reduce_mean((w*Xt + b - yt)**2)
    gw, gb = tape.gradient(loss, [w, b])            # backprop = automatic gradients
    w.assign_sub(lr*gw); b.assign_sub(lr*gb)
print(f"autodiff result: w≈{float(w):.3f}, b≈{float(b):.3f}")


## 5. Looking back at the MNIST model — every line explained
- `layers.Dense(512, activation="relu")` → learnable `relu(W@x + b)`, output size 512.
- `layers.Dense(10, activation="softmax")` → 10 outputs turned into probabilities (sum=1).
- `optimizer="adam"` → a smarter SGD (adapts the step size per weight).
- `loss="sparse_categorical_crossentropy"` → classification loss for **integer** labels
  (use `categorical_crossentropy` if labels are one-hot; `binary_crossentropy` for 2-class).
- `metrics=["accuracy"]` → what you watch (not what's optimized).
- preprocess `reshape + /255 + float32` → model wants flat float inputs in a small range.
- `fit(..., epochs, batch_size)` → runs the training loop from §4, batch by batch.
- `evaluate` on the test set → honest score on unseen data. Train≫test ⇒ **overfitting** (Ch.5).

### Loss / final-layer cheat sheet (memorize)
| problem | last layer | loss |
|---|---|---|
| binary classification | `Dense(1, "sigmoid")` | `binary_crossentropy` |
| multiclass, integer labels | `Dense(k, "softmax")` | `sparse_categorical_crossentropy` |
| multiclass, one-hot labels | `Dense(k, "softmax")` | `categorical_crossentropy` |
| regression | `Dense(1)` (no activation) | `mse` |


---
# ✍️ PROBLEMS (no solutions — that's the point)

### P1 — Tensor shapes (no code or pure numpy)
State the rank and shape for each, then verify with a numpy array:
1. a batch of 64 RGB images, 32×32 → ?
2. 500 documents, each a 20000-length word-count vector → ?
3. 250 trading days, 390 minutes each, 3 values per minute → ?

In [ ]:
import numpy as np
# TODO: build a dummy array of each shape with np.zeros(...) and print .ndim, .shape


### P2 — Manual gradient descent
Modify the §4 by-hand loop to fit `y = -3x + 0.5`. Try `lr = 0.01, 0.1, 1.5`.
Which converges? Which diverges (loss explodes)? One sentence: why does too-large `lr` blow up?

In [ ]:
# TODO


### P3 — Build & train your own net on MNIST
Without copying §1 verbatim:
1. Build a `Sequential` with **two** hidden `Dense` layers (e.g. 256 then 128, relu) + softmax output.
2. Compile, fit 5 epochs, report test accuracy.
3. Print the model's `.summary()` and read off the total parameter count.
4. Pick one test image it gets **wrong**; print predicted vs true and show it with `plt.imshow`.

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Keras classification skeleton

In [ ]:
import os; os.environ.setdefault("KERAS_BACKEND","tensorflow")
import keras
from keras import layers

model = keras.Sequential([
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax"),   # <- set NUM_CLASSES
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",  # integer labels
              metrics=["accuracy"])
model.fit(X_train, y_train, epochs=10, batch_size=128, validation_split=0.2)
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print("test acc:", acc)
model.summary()


### T2 — Image preprocessing (flatten + scale)

In [ ]:
# images shape (N, H, W) uint8 0..255  ->  (N, H*W) float32 0..1
X_train = X_train.reshape(len(X_train), -1).astype("float32") / 255.0
X_test  = X_test.reshape(len(X_test),  -1).astype("float32") / 255.0


### T3 — Manual gradient descent (pure numpy, when you need to *show* the math)

In [ ]:
import numpy as np
w, b, lr = 0.0, 0.0, 0.1
for step in range(200):
    pred = w*X + b
    grad_w = np.mean(2*(pred - y)*X)
    grad_b = np.mean(2*(pred - y))
    w -= lr*grad_w; b -= lr*grad_b


### T4 — Autodiff training loop (TensorFlow GradientTape)

In [ ]:
import tensorflow as tf
w = tf.Variable(0.0); b = tf.Variable(0.0); lr = 0.1
for step in range(200):
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean((w*Xt + b - yt)**2)
    gw, gb = tape.gradient(loss, [w, b])
    w.assign_sub(lr*gw); b.assign_sub(lr*gb)


---
### ✅ Checklist
- [ ] Give rank/shape/dtype for any data tensor; know axis 0 = samples.
- [ ] Explain `output = relu(W@x + b)` and what broadcasting/dot do.
- [ ] Write gradient descent from scratch and explain lr too-big → divergence.
- [ ] Pick the right last-layer + loss from the cheat sheet for any task.
- [ ] Build/compile/fit/evaluate a Keras classifier and read `.summary()`.

**Next: Chapter 3** — TensorFlow/PyTorch/JAX + a deeper dive on the Keras API (layers, models, the
`fit`/`compile` internals). Say "Chapter 3".